# Module 5: Pipeline Multimodal (Texte + Numerique)

Objectif : Fusionner les features textuelles (`Rapport_Collecte`) et les features numeriques au sein d'un meme pipeline grace a l'outil `ColumnTransformer` de scikit-learn, garantissant une reproductibilite parfaite.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

DATA_DIR = Path("..") / "data" / "processed"

# On charge les datasets entraines / testes precedemment
train_df = pd.read_csv(DATA_DIR / "train_clean.csv")
test_df = pd.read_csv(DATA_DIR / "test_clean.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (6449, 16)
Test shape: (1382, 16)


## 1. Preparation et separation des colonnes

On separe la cible (`Categorie`), la colonne textuelle (`Rapport_Collecte`) et le reste des variables qui sont numeriques. On doit aussi gérer les potentiels `NaN` dans la partie texte.

In [2]:
target_col = "Categorie"
text_col = "Rapport_Collecte"

# Remplacer les eventuels string manquants par du vide pour eviter les plantages NLP
train_df[text_col] = train_df[text_col].fillna("")
test_df[text_col] = test_df[text_col].fillna("")

# Identifier les colonnes numeriques (et on exclut nos cibles)
num_cols = [c for c in train_df.select_dtypes(include=["number"]).columns 
            if c not in ["Prix_Revente"]]

y_train = train_df[target_col]
X_train = train_df[[text_col] + num_cols]

y_test = test_df[target_col]
X_test = test_df[[text_col] + num_cols]

print("Text column:", text_col)
print("Numeric columns:", num_cols)

Text column: Rapport_Collecte
Numeric columns: ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite', 'Source_Centre_Tri', 'Source_Collecte_Citoyenne', 'Source_Usine_A', 'Source_Usine_B', 'Source_nan', 'Densite', 'Cond_Opacite_Ratio', 'Log_Volume']


## 2. Construction du ColumnTransformer

On va traiter le texte avec un `TfidfVectorizer` et ne rien faire (ou repasser un `StandardScaler`) pour les données numériques (qui sont d'ailleurs déjà standardisées, mais c'est propre de l'architecturer ici avec `passthrough`).

In [3]:
# Pipeline simple multimodal
preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(max_features=1000, ngram_range=(1, 2)), text_col),
        ("num", "passthrough", num_cols)
    ]
)

# On associe le preprocesseur avec le modele de classification optimal existant
clf_multi = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42))
])

# Option : ponderation / stacking (voir section 3)


## 3. Entrainement et Evaluation

On entraine le modele multimodal contenant à la fois texte et valeurs numériques pour prédire la catégorie.

In [4]:
# Entrainement
clf_multi.fit(X_train, y_train)

# Predictions sur le set de test final
preds = clf_multi.predict(X_test)

print("Pipeline Multimodal:")
print("Accuracy:", accuracy_score(y_test, preds))
print("\nClassification Report:\n", classification_report(y_test, preds))


Pipeline Multimodal:
Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

       Métal       1.00      1.00      1.00       324
      Papier       1.00      1.00      1.00       315
   Plastique       1.00      1.00      1.00       385
       Verre       1.00      1.00      1.00       358

    accuracy                           1.00      1382
   macro avg       1.00      1.00      1.00      1382
weighted avg       1.00      1.00      1.00      1382

